<a href="https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Answer:

**Lane: confirmed.** I am staying with Lane 4 (CTR opportunity scoring). My ML-02, ML-03 and ML-04 work is all built on it.

**Data:** the same frame as ML-04: March 2026 warehouse data, rows with `gsc_data_available IS TRUE`, pages with at least 100 impressions in 1–15 March and 50 in 16–31 March (75,817 pages, 37 clients). The rule uses **only 1–15 March**. The outcome `still_under` (still below typical CTR in 16–31 March) is used only to check the rule, never as an input.

### Signal checks (base rate: 35% of all pages were still under-performing in 16–31 March)

**Signal 1: CTR vs position** (the signal behind FlyRank's CTR-fix flag). Verdict: **CONFIRMED.**
Pages that were below the typical CTR for their position in 1–15 March stayed below in 16–31 March much more often (54% far below, 57% below) than pages that were above typical (42% above, 20% far above). Nuance: being *far* below typical was not more lasting than being slightly below, so the useful part of this signal is "below vs above", not "how far below". 7,836 pages were left out of this check because the typical CTR in their position band is 0.

**Signal 2: Volume** (the signal behind FlyRank's quick-win flag). Verdict: **CONFIRMED.**
Among pages below typical CTR, busier pages were more likely to stay below: 51% for 100–300 and 300–1k impressions, then 60% (1k–3k), 70% (3k–10k) and 74% (10k+). The rate is flat below 1,000 impressions and only rises above it. **This changed my rule:** I use a minimum of 1,000 impressions instead of the 500 used in the starter rule.

### My rule
In 1–15 March, a page is flagged for a title/meta review if it:
1. appeared in the top 20 positions (Google's first two pages, as in FlyRank's CTR-fix logic; this cutoff is a policy choice, not something I tested),
2. was seen at least 1,000 times (from signal check 2), and
3. had a lower CTR than typical pages in its position band (from signal check 1).

Flagged pages are ranked by **missed clicks**: how many more clicks they would have got at the typical CTR for their position. Unflagged pages get a score of 0.

- **Score:** missed clicks (flagged pages only)
- **Reason code:** `low_ctr_for_position`
- **Action label:** `review_title_and_meta` (unflagged pages: `no_action`)

In [1]:
%pip -q install duckdb

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# Connect to the warehouse
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Same frame as ML-04: one row per page. Clues from 1-15 March, answer from 16-31 March.
pages = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15') AS impressions_h1,
        SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '2026-03-15') AS clicks_h1,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_avg_position > 0)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_avg_position > 0), 0) AS position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS days_visible_h1,
        SUM(gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15') AS impressions_h2,
        SUM(gsc_clicks)      FILTER (WHERE report_date > DATE '2026-03-15') AS clicks_h2,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15' AND gsc_avg_position > 0)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15' AND gsc_avg_position > 0), 0) AS position_h2
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING impressions_h1 >= 100 AND impressions_h2 >= 50
""").df()
pages = pages.dropna(subset=["position_h1", "position_h2"]).copy()

bands = [0, 3, 5, 7, 10, 15, 20, 30, 50, 1000]

# Clues (first half only)
pages["ctr_h1"] = pages["clicks_h1"] / pages["impressions_h1"] * 100
pages["band_h1"] = pd.cut(pages["position_h1"], bands)
pages["expected_ctr_h1"] = pages.groupby("band_h1", observed=True)["ctr_h1"].transform("median")
pages["missed_clicks_h1"] = pages["impressions_h1"] * (pages["expected_ctr_h1"] - pages["ctr_h1"]) / 100

# Answer (second half only) - used to CHECK the rule, never as an input to it
pages["ctr_h2"] = pages["clicks_h2"] / pages["impressions_h2"] * 100
pages["band_h2"] = pd.cut(pages["position_h2"], bands)
pages["expected_ctr_h2"] = pages.groupby("band_h2", observed=True)["ctr_h2"].transform("median")
pages["still_under"] = (pages["ctr_h2"] < pages["expected_ctr_h2"]).astype(int)

print("Pages:", len(pages), "| clients:", pages["client_hash_id"].nunique())
print("Base rate (share still under-performing in 16-31 March):", round(pages["still_under"].mean(), 2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages: 75817 | clients: 37
Base rate (share still under-performing in 16-31 March): 0.35


In [2]:
# SIGNAL CHECK 1 - CTR vs position (the signal behind FlyRank's CTR-fix flag)
# Question: pages below typical CTR for their position in 1-15 March -
#           were they still below typical in 16-31 March?
s1 = pages[pages["expected_ctr_h1"] > 0].copy()
print("Pages left out (typical CTR for their position band is 0):", (pages["expected_ctr_h1"] == 0).sum())

s1["ctr_vs_typical"] = s1["ctr_h1"] / s1["expected_ctr_h1"]
s1["bucket"] = pd.cut(
    s1["ctr_vs_typical"], [-0.01, 0.5, 1.0, 1.5, np.inf],
    labels=["far below typical (<0.5x)", "below typical (0.5-1x)",
            "above typical (1-1.5x)", "far above typical (>1.5x)"])

table1 = s1.groupby("bucket", observed=True).agg(
    n=("still_under", "size"),
    still_under_rate=("still_under", "mean")).round(2)
display(table1)

Pages left out (typical CTR for their position band is 0): 7836


,n,still_under_rate
bucket,,
far below typical (<0.5x),27646,0.54
below typical (0.5-1x),6347,0.57
above typical (1-1.5x),5552,0.42
far above typical (>1.5x),28436,0.20


In [3]:
# SIGNAL CHECK 2 - Volume (the signal behind FlyRank's quick-win flag)
# Question: among pages BELOW typical CTR in 1-15 March,
#           are busier pages more likely to STAY below typical?
s2 = pages[pages["ctr_h1"] < pages["expected_ctr_h1"]].copy()

s2["bucket"] = pd.cut(
    s2["impressions_h1"], [99, 300, 1000, 3000, 10000, np.inf],
    labels=["100-300", "300-1k", "1k-3k", "3k-10k", "10k+"])

table2 = s2.groupby("bucket", observed=True).agg(
    n=("still_under", "size"),
    still_under_rate=("still_under", "mean")).round(2)
display(table2)

,n,still_under_rate
bucket,,
100-300,13623,0.51
300-1k,11099,0.51
1k-3k,6050,0.60
3k-10k,2639,0.70
10k+,577,0.74


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# THE RULE - inputs come from 1-15 March only
MIN_IMPRESSIONS = 1000   # signal check 2: persistence only rises above ~1,000 impressions
MAX_POSITION = 20        # Google's first two pages, as in FlyRank's CTR-fix logic

flagged = (
    (pages["impressions_h1"] >= MIN_IMPRESSIONS) &
    (pages["position_h1"] <= MAX_POSITION) &
    (pages["ctr_h1"] < pages["expected_ctr_h1"])
)

pages["score"] = np.where(flagged, pages["missed_clicks_h1"], 0.0).round(1)
pages["reason_code"] = np.where(flagged, "low_ctr_for_position", "none")
pages["action"] = np.where(flagged, "review_title_and_meta", "no_action")

# Rank: biggest score first
queue = pages.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

# Write the ranked queue (the outcome column is NOT included)
import os
os.makedirs("work/outputs", exist_ok=True)
csv_cols = ["rank", "client_hash_id", "content_hash_id", "impressions_h1", "clicks_h1", "ctr_h1",
            "position_h1", "expected_ctr_h1", "missed_clicks_h1", "score", "reason_code", "action"]
queue[csv_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv:", len(queue), "rows")
print("Pages flagged by the rule:", int(flagged.sum()), "of", len(pages))

# How good is the top of the list? Checked against 16-31 March (never used by the rule)
print("\nBase rate (random pick):", round(pages["still_under"].mean(), 2))
for k in [10, 20, 50, 100]:
    print(f"Precision@{k}: {queue['still_under'].head(k).mean():.2f}")

Wrote work/outputs/baseline_action_score.csv: 75817 rows
Pages flagged by the rule: 8710 of 75817

Base rate (random pick): 0.35
Precision@10: 0.90
Precision@20: 0.90
Precision@50: 0.88
Precision@100: 0.91


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# TOP 10 - with a little extra context from dim_content (for reading only, not used by the rule)
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

top10 = queue.head(10).copy()
id_list = "', '".join(top10["content_hash_id"])
info = con.sql(f"""
    SELECT client_hash_id, content_hash_id, content_type, main_intent
    FROM {CONTENT}
    WHERE content_hash_id IN ('{id_list}')
""").df()
top10 = top10.merge(info, on=["client_hash_id", "content_hash_id"], how="left")
top10["client"] = top10["client_hash_id"].str[-6:]   # short pseudonym, just to spot repeats

show = ["rank", "client", "content_type", "main_intent", "impressions_h1", "clicks_h1", "ctr_h1",
        "expected_ctr_h1", "position_h1", "days_visible_h1", "missed_clicks_h1", "action", "still_under"]
display(top10[show].round(2))
print("\nDifferent clients in the top 10:", top10["client_hash_id"].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rank,client,content_type,main_intent,impressions_h1,clicks_h1,ctr_h1,expected_ctr_h1,position_h1,days_visible_h1,missed_clicks_h1,action,still_under
0,1,f265ea,keyword article,informational,83772.0,0.0,0.00,0.25,0.11,15,205.68,review_title_and_meta,0
1,2,5e0096,keyword article,informational,73639.0,18.0,0.02,0.25,2.95,15,162.80,review_title_and_meta,1
2,3,f265ea,keyword article,commercial,58553.0,1.0,0.00,0.24,4.75,15,142.19,review_title_and_meta,1
3,4,5e0096,keyword article,transactional,52378.0,18.0,0.03,0.24,4.65,15,110.09,review_title_and_meta,1
4,5,5e0096,keyword article,commercial,86860.0,51.0,0.06,0.17,5.95,15,93.17,review_title_and_meta,1
5,6,9f63c4,keyword article,informational,55680.0,15.0,0.03,0.17,6.68,15,77.41,review_title_and_meta,1
6,7,5e0096,keyword article,transactional,83715.0,133.0,0.16,0.24,3.52,15,71.72,review_title_and_meta,1
7,8,5e0096,keyword article,transactional,49314.0,2.0,0.00,0.14,8.42,15,67.95,review_title_and_meta,1
8,9,5e0096,keyword article,transactional,27715.0,0.0,0.00,0.24,4.45,15,67.77,review_title_and_meta,1
9,10,043229,keyword article,informational,33020.0,14.0,0.04,0.25,1.60,13,67.07,review_title_and_meta,1



Different clients in the top 10: 4


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.